In [ ]:
import os
from Scope.Read_Write import load_binary

In [ ]:
## In this tutorial, we will discuss the STATE class. What is it exactly, and how it interacts with SPECIES and COMPUTATIONS.
## The STATE class is probably the least-intuitive class, so lets see a practical example.

In [ ]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data"
data_folder = os.path.abspath('.')+'/Data/'
## Loads the System object from a binary file, provided in the tutorial folder
sys = load_binary(f"{data_folder}ABITEM.npy")

In [ ]:
## All objects in SCOPE have a __repr__ method, so printing shows a summary of the object
print(sys)

In [ ]:
## Sources can be selected using the find_source function, and using the source name
found, source = sys.find_source("ref_hs_mol")
print(found)
print(source)

## States

In [ ]:
## Sources have "STATES"
source.states

In [ ]:
## These are accessible directly from the list
initial_state = source.states[0]
## Or through the find_state function
found, initial_state = source.find_state("initial")
print(found)
print(initial_state)

In [ ]:
## Normally, the "initial" state contains the information of the source when initiated. 
print(initial_state)
## Notice that the coordinates, for example, are just the same
print(source.coord[0])
print(initial_state.coord[0])


In [ ]:
## Computations take input information from states, and either (i) contribute new output information to this same state, or (ii) create new states.
found, final_state = source.find_state("pbe_opt")
print(final_state)
## Notice that now the coordinates are different.
print(source.coord[0])
print(final_state.coord[0])

In [ ]:
# The final_state contains the coordinates of the molecule after undergoing the following computations:
print(final_state.computations)

In [ ]:
## In this case, only one computation contributed to the final state.
comp = final_state.computations[0]
## Lets visualize the evolution of the coordinates as a result of this PBE optimization computation.
## The entire output of computations is not saved by default in the computation-class objects. It can be done, but then they become too heavy.
## So we need to re-read the output. Outputs are provided in the Sources Folder, but before, we need to overwrite the path of the computation,.
comp.out_path = f"{data_folder}ABITEM_opt_r1_HS.log" 
## Create output, which reads all lines of the output file.
output = comp.create_output()


In [ ]:
## Here, we extract all geometries.
labels, geoms = output.get_all_geometries()
## This is the evolution of the X coordinate of the first atom along the optimization 
xs = [g[0][0] for g in geoms]

## We plot the evolution of this coordinates, and show that the first step corresponds to the "initial state" coordinates...
## ...and that the last step corresponds to the "final state" 
import matplotlib.pyplot as plt
plt.plot(xs)
plt.axhline(y=initial_state.coord[0][0], color='blue', linestyle='--', linewidth=1, label='Initial State')
plt.axhline(y=final_state.coord[0][0], color='red', linestyle='--', linewidth=1, label='Final State')
plt.legend()
plt.show()

In [ ]:
## If multiple computations target the same final state, these keep updating the final state.

## For instance, when multiple optimization computations are necessary:
   ## Computation 1: Reads initial state (coordinates "A"), optimization runs, and last-step coordinates ("B") are used to create the final state.
   ## Computation 2: Reads set of coordinates "B", continues optimization, and last-step coordinates ("C") update the final state.
   ## ... This goes on until the final coordinates are considered good, or the optimization job stops.

## Why are States necessary?

In [ ]:
## For two reasons:

In [ ]:
# (1) When a computation finishes, the final information may be different than the that of the source. 
# For instance, a geometry optimization might break a bond, so the initial molecule is no longer the same. 

# Thus, if the user wants to get the molecules out of a STATEs coordinates, they need to be re-computed:
new_molec      = final_state.get_moleclist()[0]
# Then, the user can operate with the molecule-class functions regularly

## In most cases, they will be the same as the source
original_molec = source
## Species can be compared based on some compositional criteria (see specie class, function __eq__())
print(original_molec == new_molec)


In [ ]:
# (2) Because it enables the user to store multiple results of the molecule (e.g. the energy, coordinates).
# for instance, multiple optimization jobs with different DFT functionals:
found, b3lyp_state = source.find_state("b3lyp_opt")
  
print(final_state.results) # final_state is the result of a pbe optimization
print(b3lyp_state.results)

In [ ]:
# The b3lyp state is completed after two computations...
print(len(b3lyp_state.computations))

# Associated with two different jobs, with keywords:
for comp in b3lyp_state.computations:
    print(comp._job.keyword)

In [ ]:
# The second one computes the vibrational normal modes, and stores them:
print(b3lyp_state.VNMs[0])

# And this is the resulting IR spectrum
from Scope.Classes_QC import plot_ir_spectrum
x, spec = plot_ir_spectrum(b3lyp_state.VNMs)